In [ ]:
import os

os.environ["PYKEOPS_VERBOSE"] = "0"
os.environ["KEOPS_VERBOSE"] = "0"

In [ ]:
import torch

# Force CUDA re-initialisation
if torch.cuda.is_available():
    torch.cuda.init()

print(torch.cuda.device_count())  # should now be > 0

In [ ]:
import sys
sys.path.append(os.path.join(os.getcwd(), '../../')) # Add root of repo to import MBM

import warnings
import massbalancemachine as mbm
import logging
from datetime import datetime
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import json 

from regions.TF_Europe.scripts.config_TF_Europe import *
from regions.TF_Europe.scripts.dataset import *
from regions.TF_Europe.scripts.plotting import *
from regions.TF_Europe.scripts.models import *
from regions.TF_Europe.scripts.utils import *

warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2

cfg = mbm.EuropeTFConfig()
mbm.utils.seed_all(cfg.seed)
mbm.utils.free_up_cuda()
mbm.plots.use_mbm_style()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')

print(torch.cuda.is_available())   # False
print(torch.cuda.device_count())   # 0  ← wrong$

if torch.cuda.device_count()==0:
    print("No GPU detected. Using CPU.")
    device = torch.device("cpu")


In [ ]:
BASE_DIR = Path(cfg.dataPath) / path_cache / f"FD_MODEL_"
CACHE_DIR = BASE_DIR / "FD_MODEL_cache/"

# make sure BASE_DIR exists
os.makedirs(BASE_DIR, exist_ok=True)

# make sure BASE_DIR exists
os.makedirs(CACHE_DIR, exist_ok=True)

print(f"Base directory for data: {BASE_DIR}")
print(f"Cache directory for models: {CACHE_DIR}")

In [ ]:
MONTHLY_COLS = [
    't2m',
    'tp',
    'slhf',
    'sshf',
    'ssrd',
    'fal',
    'str',
    'ELEVATION_DIFFERENCE',
]
STATIC_COLS = ['aspect', 'slope', 'svf']

feature_columns = MONTHLY_COLS + STATIC_COLS

# Transform data to monthly format (run or load data):
paths = {
    'era5_climate_data':
    os.path.join(cfg.dataPath, path_ERA5_raw,
                 "era5_monthly_averaged_data_EU_US_CANADA.nc"),
    'geopotential_data':
    os.path.join(cfg.dataPath, path_ERA5_raw,
                 "era5_geopotential_pressure_EU_US_CANADA.nc")
}

# Check that all these files exists
for key, path in paths.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f"Required file for {key} not found at {path}")

vois_climate = [
    "t2m",
    "tp",
    "slhf",
    "sshf",
    "ssrd",
    "fal",
    "str",
]

vois_topographical = ["aspect", "slope", "svf"]

## Datasets:

In [ ]:
"""
Examples of loading data:
# Load Switzerland only
df = load_stakes(cfg, "CH")

# Load all Central Europe (FR+CH+IT+AT when you add them)
df_ceu = load_stakes_for_rgi_region(cfg, "11")

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}
"""

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}

rgi_regions = dfs.keys()
num_measurements = {}
for rgi_region in rgi_regions:
    src_code_regions = dfs[rgi_region].SOURCE_CODE.unique()
    for src_code in src_code_regions:
        num_measurements[src_code] = len(
            dfs[rgi_region][dfs[rgi_region].SOURCE_CODE == src_code])
num_measurements

In [ ]:
# add central asia subregions:
import geopandas as gpd

rgi_gdf = gpd.read_file(cfg.dataPath / RGI_V6_ROOT /
                        RGI_REGIONS['13']['folder'] /
                        RGI_REGIONS['13']['file'])[['RGIId', 'O2Region']]

glacier_dict = (dfs['13'][['GLACIER', 'RGIId']].drop_duplicates().merge(
    rgi_gdf, on='RGIId', how='left').set_index('GLACIER').to_dict('index'))

dfs['13']['O2Region'] = dfs['13']['GLACIER'].map({
    k: v['O2Region']
    for k, v in glacier_dict.items()
})

dfs['13'].groupby('O2Region').size().rename('n_measurements')

In [ ]:
dfs['13'].groupby('O2Region').agg(n_measurements=('GLACIER', 'size'),
                                  glaciers=('GLACIER',
                                            lambda x: sorted(x.unique())))

In [ ]:
# Build O2Region -> color mapping
o2_regions = sorted(
    set(v['O2Region'] for v in glacier_dict.values()
        if v['O2Region'] is not None))
region_colors = [
    "red", "blue", "green", "purple", "orange", "darkred", "cadetblue",
    "darkgreen", "darkpurple", "pink", "gray", "black"
]
o2_color_map = {
    r: region_colors[i % len(region_colors)]
    for i, r in enumerate(o2_regions)
}

# Map each glacier to its O2Region color
color_map = {
    glacier: o2_color_map[info['O2Region']]
    for glacier, info in glacier_dict.items() if info['O2Region'] is not None
}

m = plot_stakes_folium(dfs['13'], color_map=color_map, tooltip_col='O2Region')
m

In [ ]:
SOURCE_REGIONS = [  # regions for "foundation model"
    'CH',
    'NOR',
    'ISL',
    'FR',
]
TARGET_REGIONS = ['SJM', 'CENTRALASIA', 'IT_AT', 'ALA']

paths_multi = {
    "EU_US_CANADA": {
        "era5_climate_data":
        os.path.join(cfg.dataPath, path_ERA5_raw,
                     "era5_monthly_averaged_data_EU_US_CANADA.nc"),
        "geopotential_data":
        os.path.join(cfg.dataPath, path_ERA5_raw,
                     "era5_geopotential_pressure_EU_US_CANADA.nc"),
    },
    "HMA": {
        "era5_climate_data":
        os.path.join(cfg.dataPath, path_ERA5_raw,
                     "era5_monthly_averaged_data_HIGH_MOUNTAIN_ASIA.nc"),
        "geopotential_data":
        os.path.join(cfg.dataPath, path_ERA5_raw,
                     "era5_geopotential_pressure_HIGH_MOUNTAIN_ASIA.nc"),
    },
}

run_flag_by_code = {
    'CH': False,
    'NOR': False,
    'ISL': False,
    'FR': False,
    'SJM': False,
    'CENTRALASIA': False,
    'IT_AT': False,
    'ALA': False
}

monthly_cache = build_monthly_cache(
    cfg=cfg,
    dfs=dfs,
    paths_multi=paths_multi,
    vois_climate=vois_climate,
    vois_topographical=vois_topographical,
    region_codes=SOURCE_REGIONS + TARGET_REGIONS,
    run_flag_by_code=run_flag_by_code,
)

res_xreg_by_source = {
    region: monthly_cache[region]
    for region in SOURCE_REGIONS + TARGET_REGIONS
}

months_head_pad = res_xreg_by_source["CH"]['months_head_pad']
months_tail_pad = res_xreg_by_source["CH"]['months_tail_pad']
print(f"months_head_pad: {months_head_pad}")
print(f"months_tail_pad: {months_tail_pad}")

In [ ]:
IT_GLACIERS = [
    'CAMPO SETT.',
    'CARESER',
    'CARESER CENTRALE',
    'CARESER OCCIDENTALE',
    'CARESER ORIENTALE',
    'CIARDONEY',
    'FONTANA BIANCA / WEISSBRUNNF.',
    'GRAND ETRET',
    'LUNGA (VEDRETTA) / LANGENF.',
    'LUPO',
    'MALAVALLE (VEDR. DI) / UEBELTALF.',
    'PENDENTE (VEDR.) / HANGENDERF.',
    'RIES OCC. (VEDR. DI) / RIESERF. WESTL.',
    'SURETTA MERIDIONALE',
]

data_IT = monthly_cache['IT_AT']['data_monthly'][monthly_cache['IT_AT'][
    'data_monthly']['GLACIER'].isin(IT_GLACIERS)].copy()

res_xreg_by_source = {
    **res_xreg_by_source,
    'IT': {
        'data_monthly':
        data_IT,
        'data_monthly_aug':
        res_xreg_by_source['IT_AT']['data_monthly_aug'][res_xreg_by_source[
            'IT_AT']['data_monthly_aug']['GLACIER'].isin(IT_GLACIERS)],
        'months_head_pad':
        res_xreg_by_source['IT_AT']['months_head_pad'],
        'months_tail_pad':
        res_xreg_by_source['IT_AT']['months_tail_pad'],
    },
}

# Add O2Region to CENTRALASIA monthly data
data_ca = monthly_cache['CENTRALASIA']['data_monthly'].copy()
data_ca['O2Region'] = data_ca['GLACIER'].map({
    k: v['O2Region']
    for k, v in glacier_dict.items()
})

# Split
data_CA_12 = data_ca[data_ca['O2Region'].isin(['1', '2'])].copy()
data_CA_3 = data_ca[data_ca['O2Region'] == '3'].copy()
data_CA_4 = data_ca[data_ca['O2Region'] == '4'].copy()

print(f"Number of measurements in O2Region 1+2: {len(data_CA_12)}")
print(f"Number of measurements in O2Region 3: {len(data_CA_3)}")
print(f"Number of measurements in O2Region 4: {len(data_CA_4)}")

ca_glaciers_12 = set(data_CA_12['GLACIER'].unique())
ca_glaciers_3 = set(data_CA_3['GLACIER'].unique())
ca_glaciers_4 = set(data_CA_4['GLACIER'].unique())

res_xreg_by_source = {
    **res_xreg_by_source,
    'CA_12': {
        'data_monthly':
        data_CA_12,
        'data_monthly_aug':
        res_xreg_by_source['CENTRALASIA']['data_monthly_aug']
        [res_xreg_by_source['CENTRALASIA']['data_monthly_aug']['GLACIER'].isin(
            ca_glaciers_12)],
        'months_head_pad':
        res_xreg_by_source['CENTRALASIA']['months_head_pad'],
        'months_tail_pad':
        res_xreg_by_source['CENTRALASIA']['months_tail_pad'],
    },
    'CA_3': {
        'data_monthly':
        data_CA_3,
        'data_monthly_aug':
        res_xreg_by_source['CENTRALASIA']['data_monthly_aug']
        [res_xreg_by_source['CENTRALASIA']['data_monthly_aug']['GLACIER'].isin(
            ca_glaciers_3)],
        'months_head_pad':
        res_xreg_by_source['CENTRALASIA']['months_head_pad'],
        'months_tail_pad':
        res_xreg_by_source['CENTRALASIA']['months_tail_pad'],
    },
    'CA_4': {
        'data_monthly':
        data_CA_4,
        'data_monthly_aug':
        res_xreg_by_source['CENTRALASIA']['data_monthly_aug']
        [res_xreg_by_source['CENTRALASIA']['data_monthly_aug']['GLACIER'].isin(
            ca_glaciers_4)],
        'months_head_pad':
        res_xreg_by_source['CENTRALASIA']['months_head_pad'],
        'months_tail_pad':
        res_xreg_by_source['CENTRALASIA']['months_tail_pad'],
    },
}

TARGET_REGIONS = ['SJM', 'CENTRALASIA', 'IT', 'ALA', 'CA_12', 'CA_3', 'CA_4']

### Model assets:

#### Pooled set:

In [ ]:
# Build glacier→region mapping from all source regions (no ranking needed)
df_all_glaciers = pd.concat(
    [
        res_xreg_by_source[region]["data_monthly"][[
            "GLACIER"
        ]].drop_duplicates().assign(region=region) for region in SOURCE_REGIONS
    ],
    ignore_index=False).rename(columns={"GLACIER": "glacier"})

# Add measurement counts (needed by build_assets_from_glacier_list internally? No — only used by build_glacier_subsets)
# df_ranked just needs columns: "glacier", "region"  ← that's all build_assets uses

glaciers_all = df_all_glaciers["glacier"].tolist()
print(f"Total unique glaciers across all source regions: {len(glaciers_all)}")

assets_full = build_assets_from_glacier_list(
    glaciers=glaciers_all,
    df_ranked=df_all_glaciers,  # acts as the gl→region lookup
    res_xreg_by_source=res_xreg_by_source,
    monthly_cols=MONTHLY_COLS,
    static_cols=STATIC_COLS,
    cfg=cfg,
    cache_path=str(CACHE_DIR / "assets_foundation_100pct.joblib"),
    force_recompute=False,
    months_head_pad=months_head_pad,
    months_tail_pad=months_tail_pad,
    src_regions=SOURCE_REGIONS,
)

print(f"Foundation model assets: {len(assets_full['ds_train'])} sequences")

#### Test regions:

In [ ]:
RECOMPUTE_SPLITS = False
RECOMPUTE_REGIONS = [
]  # add regions here to recompute individually; [] to skip

save_path = BASE_DIR / "glacier_splits_FD.json"

# ── Always compute scalers/blurs — needed for any recomputation ───────────────
res_source_only = {
    region: {
        "df_train": res_xreg_by_source[region]["data_monthly"],
        "df_test": res_xreg_by_source[region]["data_monthly_aug"],
    }
    for region in SOURCE_REGIONS
}

scaler_m, scaler_s, scaler_joint = build_global_scalers_multi_source_simple(
    res_source_only,
    monthly_cols=MONTHLY_COLS,
    static_cols=STATIC_COLS,
)
blur_m, blur_s, blur_joint = estimate_global_bandwidths_simple(
    res_source_only,
    monthly_cols=MONTHLY_COLS,
    static_cols=STATIC_COLS,
    scaler_m=scaler_m,
    scaler_s=scaler_s,
    seed=cfg.seed,
)
print(
    f"Estimated blurs: blur_m={blur_m:.4f}, blur_s={blur_s:.4f}, blur_joint={blur_joint:.4f}"
)

# ── Load existing splits from disk ────────────────────────────────────────────
splits = {}
if not RECOMPUTE_SPLITS and save_path.exists():
    with open(save_path, "r") as f:
        output = json.load(f)
    splits = {
        region: {
            "pool_glaciers": data["pool_glaciers"],
            "holdout_glaciers": data["holdout_glaciers"],
            "actual_pool_frac": data["actual_pool_frac"],
            "sinkhorn_pool_vs_holdout": data["sinkhorn_pool_vs_holdout"],
            "sinkhorn_pool_vs_region": data["sinkhorn_pool_vs_region"],
            "sinkhorn_holdout_vs_region": data["sinkhorn_holdout_vs_region"],
            "blur_joint": data["blur_joint"],
            "best_seed": data["best_seed"],
            "best_score": data["best_score"],
        }
        for region, data in output.items()
    }
    print(f"Loaded splits from: {save_path}")
    for region, split in splits.items():
        print(f"\n{region}:")
        print(f"  Pool glaciers    : {split['pool_glaciers']}")
        print(f"  Holdout glaciers : {split['holdout_glaciers']}")
        print(f"  Pool fraction    : {split['actual_pool_frac']:.1%}")
        print(f"  Sk(pool↔holdout) : {split['sinkhorn_pool_vs_holdout']:.4f}")

# ── Compute splits for new/forced regions ─────────────────────────────────────
regions_to_compute = ([r for r in TARGET_REGIONS
                       if r != 'SJM'] if RECOMPUTE_SPLITS else
                      [r for r in RECOMPUTE_REGIONS if r != 'SJM'])

for region in regions_to_compute:
    print(f"\n{'='*50}")
    print(f"Splitting region: {region}")

    split = split_pool_holdout_sinkhorn(
        df_region=res_xreg_by_source[region]["data_monthly"],
        monthly_cols=MONTHLY_COLS,
        static_cols=STATIC_COLS,
        scaler_m=scaler_m,
        scaler_s=scaler_s,
        blur_joint=blur_joint,
        pool_frac=0.2,
        n_restarts=50,
        seed=cfg.seed,
    )
    splits[region] = split

    print(f"  Pool glaciers    : {split['pool_glaciers']}")
    print(f"  Holdout glaciers : {split['holdout_glaciers']}")
    print(f"  Pool fraction    : {split['actual_pool_frac']:.1%}")
    print(f"  Sk(pool↔holdout) : {split['sinkhorn_pool_vs_holdout']:.4f}")
    print(f"  Sk(pool↔region)  : {split['sinkhorn_pool_vs_region']:.4f}")
    print(f"  Sk(holdout↔reg)  : {split['sinkhorn_holdout_vs_region']:.4f}")

# ── Save back to disk (merges loaded + newly computed) ────────────────────────
if regions_to_compute:
    output = {
        region: {
            "pool_glaciers": split["pool_glaciers"],
            "holdout_glaciers": split["holdout_glaciers"],
            "actual_pool_frac": split["actual_pool_frac"],
            "sinkhorn_pool_vs_holdout": split["sinkhorn_pool_vs_holdout"],
            "sinkhorn_pool_vs_region": split["sinkhorn_pool_vs_region"],
            "sinkhorn_holdout_vs_region": split["sinkhorn_holdout_vs_region"],
            "blur_joint": split["blur_joint"],
            "best_seed": split["best_seed"],
            "best_score": split["best_score"],
        }
        for region, split in splits.items()
    }
    with open(save_path, "w") as f:
        json.dump(output, f, indent=2)
    print(f"\nSaved {len(regions_to_compute)} new split(s) to: {save_path}")
    print(f"Total regions in file: {list(output.keys())}")

In [ ]:
FT_FRAC = 0.10
FORCE_RECOMPUTE = False
FORCE_RECOMPUTE_REGIONS = [
    'CA_12', 'CA_3', 'CA_4'
]  # recompute cache for specific regions; [] to skip

ft_assets_glacier = {}  # glacier-level split
ft_assets_id = {}  # ID-level split

gl_level_split = ['CENTRALASIA', 'IT', 'ALA']
id_level_split = ['SJM', 'CENTRALASIA', 'IT', 'ALA', 'CA_12', 'CA_3', 'CA_4']

for region in TARGET_REGIONS:
    print(f"\n{'='*50}")
    print(f"Building FT assets for: {region}")

    df_monthly = res_xreg_by_source[region]["data_monthly"]
    df_monthly_aug = res_xreg_by_source[region]["data_monthly_aug"]
    head_pad = res_xreg_by_source[region]["months_head_pad"]
    tail_pad = res_xreg_by_source[region]["months_tail_pad"]

    force = FORCE_RECOMPUTE or (region in FORCE_RECOMPUTE_REGIONS)

    # ================================================================
    # 1. Glacier-level split (Sinkhorn)
    # ================================================================
    if region in gl_level_split:
        pool_glaciers = splits[region]["pool_glaciers"]
        holdout_glaciers = splits[region]["holdout_glaciers"]

        print(f"  [glacier-split] Pool    : {len(pool_glaciers)} glaciers")
        print(f"  [glacier-split] Holdout : {len(holdout_glaciers)} glaciers")

        exp_key = f"FD_to_{region}"

        ds_ft_gl = build_or_load_lstm_dataset_only(
            cfg=cfg,
            key=f"{exp_key}_FT",
            df_loss=df_monthly[df_monthly["GLACIER"].isin(pool_glaciers)],
            df_full=df_monthly_aug[df_monthly_aug["GLACIER"].isin(
                pool_glaciers)],
            months_head_pad=head_pad,
            months_tail_pad=tail_pad,
            MONTHLY_COLS=MONTHLY_COLS,
            STATIC_COLS=STATIC_COLS,
            cache_dir=str(CACHE_DIR),
            force_recompute=force,
            kind="ft",
            show_progress=False,
        )
        ds_test_gl = build_or_load_lstm_dataset_only(
            cfg=cfg,
            key=f"{exp_key}_TEST",
            df_loss=df_monthly[df_monthly["GLACIER"].isin(holdout_glaciers)],
            df_full=df_monthly_aug[df_monthly_aug["GLACIER"].isin(
                holdout_glaciers)],
            months_head_pad=head_pad,
            months_tail_pad=tail_pad,
            MONTHLY_COLS=MONTHLY_COLS,
            STATIC_COLS=STATIC_COLS,
            cache_dir=str(CACHE_DIR),
            force_recompute=force,
            kind="test",
            show_progress=False,
        )

        ft_train_idx_gl, ft_val_idx_gl = mbm.data_processing.MBSequenceDataset.split_indices(
            len(ds_ft_gl), val_ratio=0.2, seed=cfg.seed)

        ft_assets_glacier[region] = {
            "ds_ft": ds_ft_gl,
            "ds_test": ds_test_gl,
            "ft_train_idx": ft_train_idx_gl,
            "ft_val_idx": ft_val_idx_gl,
        }

        print(
            f"  [glacier-split] ds_ft  : {len(ds_ft_gl)} sequences "
            f"({ds_ft_gl.iw.sum().item()} winter | {ds_ft_gl.ia.sum().item()} annual)"
        )
        print(
            f"  [glacier-split] ds_test: {len(ds_test_gl)} sequences "
            f"({ds_test_gl.iw.sum().item()} winter | {ds_test_gl.ia.sum().item()} annual)"
        )

    # ================================================================
    # 2. ID-level split
    # ================================================================
    if region in id_level_split:
        winter_ids = df_monthly[df_monthly["PERIOD"] ==
                                "winter"]["ID"].unique().tolist()
        annual_ids = df_monthly[df_monthly["PERIOD"] ==
                                "annual"]["ID"].unique().tolist()

        rng = np.random.default_rng(cfg.seed)
        rng.shuffle(winter_ids)
        rng.shuffle(annual_ids)

        n_ft_w = max(1, int(len(winter_ids) * FT_FRAC)) if winter_ids else 0
        n_ft_a = max(1, int(len(annual_ids) * FT_FRAC))

        ft_ids = winter_ids[:n_ft_w] + annual_ids[:n_ft_a]
        test_ids = winter_ids[n_ft_w:] + annual_ids[n_ft_a:]

        print(
            f"  [ID-split] FT  : {n_ft_w} winter + {n_ft_a} annual = {len(ft_ids)} IDs"
        )
        print(f"  [ID-split] Test: {len(winter_ids)-n_ft_w} winter + "
              f"{len(annual_ids)-n_ft_a} annual = {len(test_ids)} IDs")

        exp_key_id = f"FD_to_{region}_IDsplit"

        ds_ft_id = build_or_load_lstm_dataset_only(
            cfg=cfg,
            key=f"{exp_key_id}_FT",
            df_loss=df_monthly[df_monthly["ID"].isin(ft_ids)],
            df_full=df_monthly_aug[df_monthly_aug["ID"].isin(ft_ids)],
            months_head_pad=head_pad,
            months_tail_pad=tail_pad,
            MONTHLY_COLS=MONTHLY_COLS,
            STATIC_COLS=STATIC_COLS,
            cache_dir=str(CACHE_DIR),
            force_recompute=force,
            kind="ft",
            show_progress=False,
        )
        ds_test_id = build_or_load_lstm_dataset_only(
            cfg=cfg,
            key=f"{exp_key_id}_TEST",
            df_loss=df_monthly[df_monthly["ID"].isin(test_ids)],
            df_full=df_monthly_aug[df_monthly_aug["ID"].isin(test_ids)],
            months_head_pad=head_pad,
            months_tail_pad=tail_pad,
            MONTHLY_COLS=MONTHLY_COLS,
            STATIC_COLS=STATIC_COLS,
            cache_dir=str(CACHE_DIR),
            force_recompute=force,
            kind="test",
            show_progress=False,
        )

        ft_train_idx_id, ft_val_idx_id = mbm.data_processing.MBSequenceDataset.split_indices(
            len(ds_ft_id), val_ratio=0.2, seed=cfg.seed)

        ft_assets_id[region] = {
            "ds_ft": ds_ft_id,
            "ds_test": ds_test_id,
            "ft_train_idx": ft_train_idx_id,
            "ft_val_idx": ft_val_idx_id,
        }

        print(
            f"  [ID-split] ds_ft  : {len(ds_ft_id)} sequences "
            f"({ds_ft_id.iw.sum().item()} winter | {ds_ft_id.ia.sum().item()} annual)"
        )
        print(
            f"  [ID-split] ds_test: {len(ds_test_id)} sequences "
            f"({ds_test_id.iw.sum().item()} winter | {ds_test_id.ia.sum().item()} annual)"
        )

# convenience dict: glacier-split where available, ID-split for SJM
ft_assets = {**ft_assets_glacier, "SJM": ft_assets_id["SJM"]}

## Transformer:

In [ ]:
best_params_gs = {
    'Fm': 8,
    'Fs': 3,
    'd_model': 128,
    'nhead': 8,
    'num_layers': 3,
    'dim_feedforward': 128,
    'dropout': 0.1,
    'static_layers': 2,
    'static_hidden': 64,
    'static_dropout': 0.0,
    'head_dropout': 0.1,
    'lr': 0.0005,
    'weight_decay': 1e-05,
    'two_heads': False,
    'loss_spec': None,
    'T_max': 32
}

print("\nbest_params_gs:")
for k, v in best_params_gs.items():
    print(f"  {k}: {v}")

### Pooled model:

In [ ]:
TRAIN_FLAG = False
MODEL_DATE = "2026-05-28"
models_dir = BASE_DIR / "models" / "foundation"
os.makedirs(models_dir, exist_ok=True)

model_foundation, model_path, info = train_or_load_one_source_model(
    cfg=cfg,
    key="foundation_100pct",
    lstm_assets=assets_full,
    best_params=best_params_gs,
    device=device,
    models_dir=models_dir,
    prefix="transformer_foundation",
    train_flag=TRAIN_FLAG,
    force_retrain=False,
    epochs=150,
    date=MODEL_DATE,
    model_type="transformer",
    verbose=False,
)

print(f"Foundation model saved to: {model_path}")

In [ ]:
print(f"\n{'='*60}")
print("Zero-shot evaluation of foundation model on target regions")
print(f"{'='*60}")

for split_label, assets_dict, regions in [
    ("glacier-split", ft_assets_glacier, gl_level_split),
    ("ID-split", ft_assets_id, id_level_split),
]:
    print(f"\n--- {split_label} ---")
    for region in regions:
        ds_scaler = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
            assets_full["ds_train"])
        ds_scaler.make_loaders(
            train_idx=assets_full["train_idx"],
            val_idx=assets_full["val_idx"],
            fit_and_transform=True,
            seed=cfg.seed,
            verbose=False,
        )
        ds_test_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
            assets_dict[region]["ds_test"])
        test_dl = mbm.data_processing.MBSequenceDataset.make_test_loader(
            ds_test_copy,
            ds_scaler,
            batch_size=128,
            seed=cfg.seed,
        )
        metrics, _ = model_foundation.evaluate_with_preds(
            device, test_dl, ds_test_copy)
        print(f"  {region:15s}  RMSE_a={metrics['RMSE_annual']:.3f}  "
              f"RMSE_w={metrics['RMSE_winter']:.3f}  "
              f"R2_a={metrics['R2_annual']:.3f}  "
              f"R2_w={metrics['R2_winter']:.3f}")

In [ ]:
for split_label, assets_dict, regions in [
    ("glacier-split", ft_assets_glacier, gl_level_split),
    ("ID-split", ft_assets_id, id_level_split),
]:
    for region in regions:
        plot_pred_vs_truth_grid(
            plot_configs=[(f"{region} (zero-shot)", model_foundation,
                           assets_full)],
            ds_test=assets_dict[region]["ds_test"],
            device=device,
            cfg=cfg,
            ncols=1,
            ax_xlim=(-8, 6),
            ax_ylim=(-8, 6),
            title=f"Foundation model → {region} (zero-shot, {split_label})",
            save_path=f"figures/foundation_zeroshot_{region}_{split_label}",
            figsize=(5, 5),
        )

#### Compare to single shot models:

In [ ]:
# ================================================================
# Compare foundation model (zero-shot) vs single-source transformers
# on IT_AT, SJM, CENTRALASIA
# ================================================================

XREG_MODEL_DATE = "2026-05-29"
XREG_MODELS_BASE = Path(cfg.dataPath) / path_cache / "CROSS_REGION" / "models"

single_src_params = {
    'Fm': 8,
    'Fs': 3,
    'd_model': 64,
    'nhead': 8,
    'num_layers': 3,
    'dim_feedforward': 128,
    'dropout': 0.1,
    'static_layers': 1,
    'static_hidden': 128,
    'static_dropout': 0.1,
    'head_dropout': 0.1,
    'lr': 0.0005,
    'weight_decay': 1e-05,
    'two_heads': False,
    'loss_spec': None,
    'T_max': 32
}

# --- scaler donor (foundation pool) ---
ds_pooled_scaler = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
    assets_full["ds_train"])
ds_pooled_scaler.make_loaders(
    train_idx=assets_full["train_idx"],
    val_idx=assets_full["val_idx"],
    fit_and_transform=True,
    seed=cfg.seed,
    verbose=False,
)

# --- load or train single-source models ---
single_source_models = {}
for src in SOURCE_REGIONS:
    ckpt = XREG_MODELS_BASE / src / f"transformer_xreg_{src}_{src}_{XREG_MODEL_DATE}.pt"
    os.makedirs(ckpt.parent, exist_ok=True)

    m = mbm.models.Transformer_MB.build_model_from_params(cfg,
                                                          single_src_params,
                                                          device,
                                                          verbose=False)

    if ckpt.exists():
        state = torch.load(ckpt, map_location=device)
        m.load_state_dict(state)
        print(f"Loaded {src} model from {ckpt}")
    else:
        print(f"No checkpoint found for {src}, training from scratch...")
        # build assets from this source region only
        df_src = res_xreg_by_source[src]["data_monthly"][["GLACIER"]] \
                   .drop_duplicates().assign(region=src) \
                   .rename(columns={"GLACIER": "glacier"})
        assets_src = build_assets_from_glacier_list(
            glaciers=df_src["glacier"].tolist(),
            df_ranked=df_src,
            res_xreg_by_source=res_xreg_by_source,
            monthly_cols=MONTHLY_COLS,
            static_cols=STATIC_COLS,
            cfg=cfg,
            cache_path=str(XREG_MODELS_BASE / src /
                           f"assets_{src}_100pct.joblib"),
            force_recompute=False,
            months_head_pad=months_head_pad,
            months_tail_pad=months_tail_pad,
            src_regions=[src],
        )
        m, _, _ = train_or_load_one_source_model(
            cfg=cfg,
            key=src,
            lstm_assets=assets_src,
            best_params=single_src_params,
            device=device,
            models_dir=XREG_MODELS_BASE / src,
            prefix=f"transformer_xreg_{src}",
            train_flag=False,
            force_retrain=False,
            epochs=150,
            date=XREG_MODEL_DATE,
            model_type="transformer",
            verbose=False,
        )
        print(f"Trained {src} model, saved to {ckpt}")

    single_source_models[src] = m

In [ ]:
# --- evaluation helper ---
def _eval_on(model, ds_test):
    ds_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
        ds_test)
    test_dl = mbm.data_processing.MBSequenceDataset.make_test_loader(
        ds_copy, ds_pooled_scaler, batch_size=128, seed=cfg.seed)
    return model.evaluate_with_preds(device, test_dl, ds_copy)


# --- collect metrics ---
eval_configs = [
    ("IT (GL-split)", ft_assets_glacier.get("IT", ft_assets_id.get("IT"))),
    ("SJM (ID-split)", ft_assets_id["SJM"]),
    ("CENTRALASIA (GL-split)", ft_assets_glacier["CENTRALASIA"]),
    ("CENTRALASIA (ID-split)", ft_assets_id["CENTRALASIA"]),
    ("ALA (GL-split)", ft_assets_glacier.get("ALA", ft_assets_id.get("ALA"))),
    ("ALA (ID-split)", ft_assets_id.get("ALA")),
    ("CA_12 (ID-split)", ft_assets_id.get("CA_12")),
    ("CA_3 (ID-split)", ft_assets_id.get("CA_3")),
    ("CA_4 (ID-split)", ft_assets_id.get("CA_4")),
]

df_metrics_comparison = {}
for tgt_label, assets in eval_configs:
    if assets is None:
        print(f"No assets for {tgt_label}, skipping.")
        continue

    df_metrics_comparison[tgt_label] = {}
    m, _ = _eval_on(model_foundation, assets["ds_test"])
    df_metrics_comparison[tgt_label]["foundation"] = m

    for src, model in single_source_models.items():
        m, _ = _eval_on(model, assets["ds_test"])
        df_metrics_comparison[tgt_label][src] = m

# --- print summary ---
print(f"\n{'='*80}")
print(f"{'Model':<15} {'RMSE_a':>8} {'RMSE_w':>8} {'R2_a':>8} {'R2_w':>8}")
print(f"{'='*80}")
for tgt_label, results in df_metrics_comparison.items():
    print(f"\n--- {tgt_label} ---")
    for model_label, m in results.items():
        print(f"  {model_label:<15}  "
              f"RMSE_a={m['RMSE_annual']:.3f}  "
              f"RMSE_w={m['RMSE_winter']:.3f}  "
              f"R2_a={m['R2_annual']:.3f}  "
              f"R2_w={m['R2_winter']:.3f}")

In [ ]:
# --- bar plot ---
MODEL_ORDER = SOURCE_REGIONS + ["foundation"]
COLORS = {
    "CH": NATURE_PALETTE["blue"],
    "NOR": NATURE_PALETTE["vermillion"],
    "ISL": NATURE_PALETTE["reddish_purple"],
    "FR": NATURE_PALETTE["orange"],
    "foundation": "black",
}
LABELS = {src: f"{src} only" for src in SOURCE_REGIONS}
LABELS["foundation"] = "Foundation\n(pooled)"
METRICS = [
    ("RMSE_annual", "RMSE annual (m w.e.)", False),
    ("RMSE_winter", "RMSE winter (m w.e.)", False),
]

def has_winter_comparison(tgt_label):
    results = df_metrics_comparison[tgt_label]
    winter_vals = [
        results[m]["RMSE_winter"]
        for m in MODEL_ORDER
        if m in results and "RMSE_winter" in results[m]
    ]
    return any(not np.isnan(v) for v in winter_vals)

def get_metrics_for_region_comparison(tgt_label):
    if not has_winter_comparison(tgt_label):
        return [(k, l, r) for k, l, r in METRICS if k != "RMSE_winter"]
    return METRICS

n_tgts = len(df_metrics_comparison)
n_metrics = len(METRICS)

fig, axes = plt.subplots(
    n_metrics, n_tgts,
    figsize=(3.5 * n_tgts, 3.8 * n_metrics),
    sharey=False,
)
axes = np.array(axes).reshape(n_metrics, n_tgts)

for col, (tgt_label, results) in enumerate(df_metrics_comparison.items()):
    region_metrics = get_metrics_for_region_comparison(tgt_label)
    for row, (metric_key, metric_label, is_r2) in enumerate(METRICS):
        ax = axes[row, col]

        if (metric_key, metric_label, is_r2) not in region_metrics:
            ax.set_visible(False)
            continue

        models = [m for m in MODEL_ORDER if m in results]
        vals = [results[m][metric_key] for m in models]
        colors = [COLORS[m] for m in models]
        bars = ax.bar(range(len(models)), vals,
                      color=colors, edgecolor="white",
                      linewidth=0.8, alpha=0.5, zorder=3)

        fidx = models.index("foundation") if "foundation" in models else None
        if fidx is not None:
            bars[fidx].set_edgecolor("black")
            bars[fidx].set_linewidth(2)
            bars[fidx].set_alpha(0.5)

        for bar, val in zip(bars, vals):
            y_pos = bar.get_height() + 0.02 if val >= 0 else bar.get_height() - 0.08
            ax.text(bar.get_x() + bar.get_width() / 2, y_pos,
                    f"{val:.2f}", ha="center", va="bottom", fontsize=7)

        if is_r2:
            ax.axhline(0, color="black", linewidth=0.8, linestyle="--", zorder=2)

        ax.set_xticks(range(len(models)))
        if row == n_metrics - 1:
            ax.set_xticklabels([LABELS[m] for m in models],
                               rotation=30, ha="right",
                               fontsize=NATURE_SPECS["font_max_pt"])
        else:
            ax.set_xticklabels([])

        if col == 0:
            ax.set_ylabel(metric_label, fontsize=NATURE_SPECS["font_max_pt"])
        if row == 0:
            ax.set_title(tgt_label, fontsize=NATURE_SPECS["font_max_pt"] + 1,
                         fontweight="bold")

        valid_vals = [v for v in vals if not np.isnan(v)]
        if is_r2:
            ymin = min(min(valid_vals) * 1.3, -0.2)
            ymax = max(max(valid_vals) * 1.3, 0.5)
        else:
            ymin = 0
            ymax = max(valid_vals) * 1.25
        ax.set_ylim(ymin, ymax)

        ax.yaxis.grid(False, linestyle="--", alpha=0.5, zorder=0)
        ax.set_axisbelow(False)
        apply_nature_style(ax, fontsize=NATURE_SPECS["font_max_pt"], box=False)

fig.suptitle("Foundation model (zero-shot) vs single-source Transformers",
             fontsize=NATURE_SPECS["font_max_pt"] + 2, fontweight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.99])
fig.savefig("figures/foundation_vs_single_source_barplot.png",
            dpi=NATURE_SPECS["dpi"], bbox_inches="tight")
plt.show()

### Fine tuning:

In [ ]:
TRAIN_FLAG = False
models_dir_ft = BASE_DIR / "models" / "finetuned"
os.makedirs(models_dir_ft, exist_ok=True)

# scaler donor: always cloned from the foundation training assets
ds_pooled_scaler = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
    assets_full["ds_train"])
ds_pooled_scaler.make_loaders(
    train_idx=assets_full["train_idx"],
    val_idx=assets_full["val_idx"],
    fit_and_transform=True,
    seed=cfg.seed,
    verbose=False,
)

ft_models_gl = {}  # glacier-split
ft_models_id = {}  # ID-split only

for split_label, assets_dict, models_dict, regions, suffix in [
    ("GLsplit", ft_assets_glacier, ft_models_gl, gl_level_split, ""),
    ("IDsplit", ft_assets_id, ft_models_id, id_level_split, "_IDsplit"),
]:
    for region in regions:
        models_dict[region] = {}
        print(f"\n{'='*60}")
        print(f"Fine-tuning on: {region} ({split_label})")

        for strategy in ["head_only", "full", "adapter"]:
            models_dict[region][strategy] = finetune_transformer_on_target(
                cfg=cfg,
                model_pooled=model_foundation,
                ds_ft=assets_dict[region]["ds_ft"],
                ds_pooled_scaler=ds_pooled_scaler,
                device=device,
                best_params=best_params_gs,
                model_filename=str(
                    models_dir_ft /
                    f"transformer_ft_{strategy}_{region}{suffix}_{MODEL_DATE}.pt"
                ),
                strategy=strategy,
                epochs=50,
                lr_factor=0.1,
                force_retrain=TRAIN_FLAG,
            )
            print(f"  [{region}] {strategy} done")

In [ ]:
source_codes_pretrain = build_source_codes_for_dataset(
    assets_full["ds_train"],
    pd.concat(
        [res_xreg_by_source[r]["data_monthly_aug"] for r in SOURCE_REGIONS],
        ignore_index=False),
    source_col="SOURCE_CODE",
)

dan_models = {}  # glacier-split
dan_models_id = {}  # ID-split

for split_label, assets_dict, dan_dict, regions, suffix in [
    ("GLsplit", ft_assets_glacier, dan_models, gl_level_split, ""),
    ("IDsplit", ft_assets_id, dan_models_id, id_level_split, "_IDsplit"),
]:
    for region in regions:
        print(f"\n{'='*50}  DAN → {region} ({split_label})")

        source_codes_ft = [region] * len(assets_dict[region]["ds_ft"])

        dan_dict[region], _ = finetune_dan_transformer_on_target(
            cfg=cfg,
            model_foundation=model_foundation,
            assets_full=assets_full,
            ft_assets_region=assets_dict[region],
            ds_pooled_scaler=ds_pooled_scaler,
            source_codes_pretrain=source_codes_pretrain,
            source_codes_ft=source_codes_ft,
            device=device,
            best_params=best_params_gs,
            model_filename=str(
                models_dir_ft /
                f"transformer_dan_{region}{suffix}_{MODEL_DATE}.pt"),
            dan_alpha=0.1,
            grl_lambda=1.0,
            mix_ratio_ft=1.0,
            epochs=60,
            lr_backbone=5e-5,
            lr_domain=1e-4,
            force_retrain=TRAIN_FLAG,
            verbose=False,
        )

In [ ]:
# --- unified evaluation ---
print(f"\n{'='*75}")
print(f"{'Strategy':<25} {'RMSE_a':>8} {'RMSE_w':>8} {'R2_a':>8} {'R2_w':>8}")
print(f"{'='*75}")

for split_label, assets_dict, models_dict, dan_dict, regions in [
    ("default", ft_assets, ft_models_gl, dan_models, gl_level_split),
    ("IDsplit", ft_assets_id, ft_models_id, dan_models_id, id_level_split),
]:
    for region in regions:
        print(f"\n--- {region} ({split_label}) ---")

        def _eval(model, region=region, assets_dict=assets_dict):
            ds_test_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
                assets_dict[region]["ds_test"])
            test_dl = mbm.data_processing.MBSequenceDataset.make_test_loader(
                ds_test_copy, ds_pooled_scaler, batch_size=128, seed=cfg.seed)
            return model.evaluate_with_preds(device, test_dl, ds_test_copy)

        all_models = {
            "no_ft": model_foundation,
            **{
                s: models_dict[region][s]
                for s in ["head_only", "adapter", "full"]
            },
            "dan": dan_dict[region],
        }

        for label, model in all_models.items():
            m, _ = _eval(model)
            print(f"  {label:<25}  "
                  f"RMSE_a={m['RMSE_annual']:.3f}  "
                  f"RMSE_w={m['RMSE_winter']:.3f}  "
                  f"R2_a={m['R2_annual']:.3f}  "
                  f"R2_w={m['R2_winter']:.3f}")

In [ ]:
for split_label, assets_dict, models_dict, dan_dict, regions in [
    ("GLsplit", ft_assets, ft_models_gl, dan_models, gl_level_split),
    ("IDsplit", ft_assets_id, ft_models_id, dan_models_id, id_level_split),
]:
    for region in regions:
        plot_configs = [
            ("no_ft (zero-shot)", model_foundation, assets_full),
            *[(strategy, models_dict[region][strategy], assets_full)
              for strategy in ["head_only", "full", "adapter"]],
            ("dan", dan_dict[region], assets_full),
        ]

        plot_pred_vs_truth_grid(
            plot_configs=plot_configs,
            ds_test=assets_dict[region]["ds_test"],
            device=device,
            cfg=cfg,
            ncols=5,
            ax_xlim=(-8, 6),
            ax_ylim=(-8, 6),
            title=
            f"Foundation model → {region}: zero-shot vs fine-tuning strategies ({split_label})",
            save_path=
            f"figures/foundation_ft_strategies_{region}_{split_label}",
            figsize=(25, 5),
        )

In [ ]:
for split_label, assets_dict, models_dict, dan_dict, regions in [
        # ("GLsplit", ft_assets, ft_models_gl, dan_models, gl_level_split),
    ("IDsplit", ft_assets_id, ft_models_id, dan_models_id, id_level_split),
]:
    for region in regions:
        plot_configs = [
            ("no_ft (zero-shot)", model_foundation, assets_full),
            *[(strategy, models_dict[region][strategy], assets_full)
              for strategy in ["full"]],
            ("dan", dan_dict[region], assets_full),
        ]

        plot_pred_vs_truth_grid(
            plot_configs=plot_configs,
            ds_test=assets_dict[region]["ds_test"],
            device=device,
            ncols=3,
            cfg=cfg,
            ax_xlim=(-8, 6),
            ax_ylim=(-8, 6),
            title=
            f"Foundation model → {region}: zero-shot vs fine-tuning strategies ({split_label})",
            save_path=
            f"figures/foundation_ft_strategies_{region}_{split_label}",
            figsize=(25, 5),
        )

##### Compare to single shot:

In [ ]:
# ================================================================
# Compare foundation FT vs single-source FT on IT, SJM, CENTRALASIA
# ================================================================

FT_STRATEGY = "full"  # toggle: "head_only", "adapter", "full"

XREG_FT_DATE = "2026-05-29"
XREG_MODELS_BASE = Path(cfg.dataPath) / path_cache / "CROSS_REGION" / "models"

# --- build single-source assets and scalers (needed for FT) ---
single_source_assets = {}
single_source_scalers = {}

for src in SOURCE_REGIONS:
    df_src = res_xreg_by_source[src]["data_monthly"][["GLACIER"]] \
               .drop_duplicates().assign(region=src) \
               .rename(columns={"GLACIER": "glacier"})
    assets_src = build_assets_from_glacier_list(
        glaciers=df_src["glacier"].tolist(),
        df_ranked=df_src,
        res_xreg_by_source=res_xreg_by_source,
        monthly_cols=MONTHLY_COLS,
        static_cols=STATIC_COLS,
        cfg=cfg,
        cache_path=str(XREG_MODELS_BASE / src / f"assets_{src}_100pct.joblib"),
        force_recompute=False,
        months_head_pad=months_head_pad,
        months_tail_pad=months_tail_pad,
        src_regions=[src],
    )
    single_source_assets[src] = assets_src

    scaler = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
        assets_src["ds_train"])
    scaler.make_loaders(
        train_idx=assets_src["train_idx"],
        val_idx=assets_src["val_idx"],
        fit_and_transform=True,
        seed=cfg.seed,
        verbose=False,
    )
    single_source_scalers[src] = scaler

# --- fine-tune each single-source model on each target ---
single_ft_models = {}  # single_ft_models[src][tgt_label]

for src in SOURCE_REGIONS:
    single_ft_models[src] = {}
    base_model = single_source_models[src]

    for tgt_label, assets in eval_configs:
        if assets is None:
            continue

        ft_dir = XREG_MODELS_BASE / src / "finetuned"
        os.makedirs(ft_dir, exist_ok=True)

        ckpt = ft_dir / f"transformer_ft_{FT_STRATEGY}_{src}_to_{tgt_label.replace(' ', '_').replace('(', '').replace(')', '')}_{XREG_FT_DATE}.pt"

        model_ft = finetune_transformer_on_target(
            cfg=cfg,
            model_pooled=base_model,
            ds_ft=assets["ds_ft"],
            ds_pooled_scaler=single_source_scalers[src],
            device=device,
            best_params=single_src_params,
            model_filename=str(ckpt),
            strategy=FT_STRATEGY,
            epochs=50,
            lr_factor=0.1,
            force_retrain=not ckpt.exists(),
        )
        single_ft_models[src][tgt_label] = model_ft
        print(f"  [{src} → {tgt_label}] {FT_STRATEGY} done")

In [ ]:
# --- collect metrics: foundation FT vs single-source FT ---
df_metrics_ft_comparison = {}

for tgt_label, assets in eval_configs:
    if assets is None:
        continue

    df_metrics_ft_comparison[tgt_label] = {}
    tgt_key = tgt_label.split()[0]
    is_id = "ID-split" in tgt_label or tgt_key == "SJM"  # SJM always ID-split
    mdict = ft_models_id if is_id else ft_models_gl

    # zero-shot foundation baseline
    m, _ = _eval_on(model_foundation, assets["ds_test"])
    df_metrics_ft_comparison[tgt_label]["foundation_zs"] = m

    # foundation fine-tuned
    if tgt_key in mdict and FT_STRATEGY in mdict[tgt_key]:
        m, _ = _eval_on(mdict[tgt_key][FT_STRATEGY], assets["ds_test"])
        df_metrics_ft_comparison[tgt_label]["foundation_ft"] = m

    # single-source FT
    for src in SOURCE_REGIONS:
        if tgt_label in single_ft_models.get(src, {}):
            m, _ = _eval_on(single_ft_models[src][tgt_label],
                            assets["ds_test"])
            df_metrics_ft_comparison[tgt_label][src] = m

# --- print summary ---
print(f"\n{'='*80}")
print(f"Strategy: {FT_STRATEGY}")
print(f"{'Model':<20} {'RMSE_a':>8} {'RMSE_w':>8} {'R2_a':>8} {'R2_w':>8}")
print(f"{'='*80}")
for tgt_label, results in df_metrics_ft_comparison.items():
    print(f"\n--- {tgt_label} ---")
    for model_label, m in results.items():
        print(f"  {model_label:<20}  "
              f"RMSE_a={m['RMSE_annual']:.3f}  "
              f"RMSE_w={m['RMSE_winter']:.3f}  "
              f"R2_a={m['R2_annual']:.3f}  "
              f"R2_w={m['R2_winter']:.3f}")

In [ ]:
# --- bar plot ---
FT_MODEL_ORDER = ["foundation_zs", "foundation_ft"] + SOURCE_REGIONS
FT_COLORS = {
    "foundation_zs": "#aaaaaa",
    "foundation_ft": "black",
    "CH": NATURE_PALETTE["blue"],
    "NOR": NATURE_PALETTE["vermillion"],
    "ISL": NATURE_PALETTE["reddish_purple"],
    "FR": NATURE_PALETTE["orange"],
}
FT_LABELS = {
    "foundation_zs": "Fnd.\n(0-shot)",
    "foundation_ft": f"Fnd.\n({FT_STRATEGY})",
    **{
        src: f"{src}\n({FT_STRATEGY})"
        for src in SOURCE_REGIONS
    },
}

METRICS = [
    ("RMSE_annual", "RMSE annual\n(m w.e.)", False),
    ("RMSE_winter", "RMSE winter\n(m w.e.)", False),
]

R2_YMIN_CLIP = -2.0

def has_winter(tgt_label):
    """Check whether a target region has any winter measurements."""
    # df_metrics_ft_comparison keys are display labels, not region codes.
    # Look up directly in res_xreg_by_source by stripping the split suffix,
    # or fall back to checking the metrics dict itself (NaN RMSE_winter = no data).
    results = df_metrics_ft_comparison[tgt_label]
    # If all models return NaN for RMSE_winter, there's no winter data
    winter_vals = [
        results[m]["RMSE_winter"]
        for m in FT_MODEL_ORDER
        if m in results and "RMSE_winter" in results[m]
    ]
    return any(not np.isnan(v) for v in winter_vals)

def get_metrics_for_region(tgt_label):
    """Return only the METRICS rows that are valid for this region."""
    if not has_winter(tgt_label):
        return [(k, l, r) for k, l, r in METRICS if k != "RMSE_winter"]
    return METRICS

n_tgts = len(df_metrics_ft_comparison)
# Use max metric count across regions to size the figure
n_metrics = len(METRICS)

fig, axes = plt.subplots(
    n_metrics, n_tgts,
    figsize=(3.5 * n_tgts, 3.8 * n_metrics),
    sharey=False,
)
axes = np.array(axes).reshape(n_metrics, n_tgts)

for col, (tgt_label, results) in enumerate(df_metrics_ft_comparison.items()):
    region_metrics = get_metrics_for_region(tgt_label)
    for row, (metric_key, metric_label, is_r2) in enumerate(METRICS):
        ax = axes[row, col]

        # Hide the axis entirely if this metric isn't valid for this region
        if (metric_key, metric_label, is_r2) not in region_metrics:
            ax.set_visible(False)
            continue

        models = [m for m in FT_MODEL_ORDER if m in results]
        vals = [results[m][metric_key] for m in models]
        colors = [FT_COLORS[m] for m in models]
        display_vals = [max(v, R2_YMIN_CLIP) if is_r2 else v for v in vals]
        valid_vals = [v for v in vals if not np.isnan(v)]
        valid_display = [v for v in display_vals if not np.isnan(v)]

        bars = ax.bar(range(len(models)), display_vals,
                      color=colors, edgecolor="white",
                      linewidth=0.8, alpha=0.75, zorder=3)

        for i, mod in enumerate(models):
            if mod.startswith("foundation"):
                bars[i].set_edgecolor("black")
                bars[i].set_linewidth(1.5)
                bars[i].set_alpha(1.0)

        if is_r2:
            ymin = max(min(valid_display) * 1.15, R2_YMIN_CLIP * 1.05)
            ymax = max(max(valid_display) * 1.15, 0.5)
        else:
            ymin = 0
            ymax = max(valid_vals) * 1.45
        ax.set_ylim(ymin, ymax)

        for bar, val, dval in zip(bars, vals, display_vals):
            clipped = is_r2 and val < R2_YMIN_CLIP
            txt = f"{val:.2f}" + ("*" if clipped else "")
            bar_top = bar.get_height()
            if bar_top > 0.78 * ymax:
                y_pos, va, color = bar_top - 0.06, "top", "white"
            elif bar_top >= 0:
                y_pos, va, color = bar_top + 0.02, "bottom", "black"
            else:
                y_pos, va, color = bar_top - 0.04, "top", "black"
            ax.text(bar.get_x() + bar.get_width() / 2, y_pos, txt,
                    ha="center", va=va, fontsize=6.5, color=color, clip_on=False)

        if is_r2:
            ax.axhline(0, color="black", linewidth=0.8, linestyle="--", zorder=2)
        if is_r2 and any(v < R2_YMIN_CLIP for v in vals):
            ax.text(0.98, 0.02, "* clipped", transform=ax.transAxes,
                    ha="right", va="bottom", fontsize=5.5,
                    color="#666666", style="italic")

        ax.set_xticks(range(len(models)))
        ax.set_xticklabels(
            [FT_LABELS[m] for m in models],
            rotation=40, ha="right",
            fontsize=max(NATURE_SPECS["font_max_pt"] - 1, 6),
        )

        if col == 0:
            ax.set_ylabel(metric_label, fontsize=NATURE_SPECS["font_max_pt"])
        if row == 0:
            ax.set_title(tgt_label, fontsize=NATURE_SPECS["font_max_pt"] + 1,
                         fontweight="bold", pad=6)

        apply_nature_style(ax, fontsize=NATURE_SPECS["font_max_pt"], box=False)

fig.suptitle(
    f"Foundation vs single-source Transformers — FT strategy: {FT_STRATEGY}",
    fontsize=NATURE_SPECS["font_max_pt"] + 2, fontweight="bold", y=1.01,
)
fig.tight_layout(rect=[0, 0, 1, 0.99])
fig.savefig(f"figures/foundation_vs_single_source_ft_{FT_STRATEGY}.png",
            dpi=NATURE_SPECS["dpi"], bbox_inches="tight")
plt.show()

## Smart finetuning:

In [ ]:
from regions.TF_Europe.scripts.dataset.topoclimatic_distance import _estimate_blur, _sinkhorn_distance

pure_static = [c for c in STATIC_COLS if c != "ELEVATION_DIFFERENCE"]


def _stake_topo(df):
    parts = [df.groupby("ID")[pure_static].first()]
    if "ELEVATION_DIFFERENCE" in STATIC_COLS:
        parts.append(df.groupby("ID")[["ELEVATION_DIFFERENCE"]].mean())
    return pd.concat(parts, axis=1)[STATIC_COLS].to_numpy(dtype=np.float64)


### On ID:

In [ ]:
FT_RANK_REGIONS = ['CENTRALASIA', 'CA_12', 'CA_3', 'CA_4', 'ALA']

res_all_regions = {
    region: {
        "df_train": res_xreg_by_source[region]["data_monthly"],
        "df_test":  res_xreg_by_source[region]["data_monthly_aug"],
    }
    for region in SOURCE_REGIONS + FT_RANK_REGIONS
}

scaler_m, scaler_s, scaler_joint = build_global_scalers_multi_source_simple(
    res_all_regions, monthly_cols=MONTHLY_COLS, static_cols=STATIC_COLS,
)
blur_m, blur_s, blur_joint = estimate_global_bandwidths_simple(
    res_all_regions, monthly_cols=MONTHLY_COLS, static_cols=STATIC_COLS,
    scaler_m=scaler_m, scaler_s=scaler_s, seed=cfg.seed,
)
print(f"Scalers and blurs ready: blur_m={blur_m:.4f}, blur_s={blur_s:.4f}, blur_joint={blur_joint:.4f}")

FRACS_FT = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35]
N_RANDOM_SEEDS = 5

In [ ]:
# ── Greedy coverage FT ID selection ──────────────────────────────────────────
def greedy_coverage_ft_selection_1(
    X_joint_all: np.ndarray,
    id_array: np.ndarray,
    ft_ids: set,
    test_ids: set,
    n_select: int,
    seed: int = 0,
) -> list:
    """
    Greedy farthest-point coreset: selects FT IDs that maximally cover
    the feature space, seeded by the ID closest to the test set centroid.
    O(n_select * n_candidates) L2 operations — runs in milliseconds.
    """
    ft_id_list = sorted(ft_ids)
    id_to_feat = {
        id_val: X_joint_all[id_array == id_val].mean(axis=0)
        for id_val in ft_id_list if (id_array == id_val).sum() >= 1
    }
    ft_id_list = list(id_to_feat.keys())
    X_ft = np.stack([id_to_feat[i] for i in ft_id_list])  # (n_ft, d)

    # Seed: FT ID closest to test set centroid
    test_mask = np.isin(id_array, list(test_ids))
    X_test_mean = X_joint_all[test_mask].mean(axis=0, keepdims=False)
    first_idx = int(np.argmin(np.linalg.norm(X_ft - X_test_mean, axis=1)))

    selected_indices = [first_idx]
    remaining_indices = [i for i in range(len(ft_id_list)) if i != first_idx]

    for _ in range(n_select - 1):
        if not remaining_indices:
            break
        X_selected = X_ft[selected_indices]
        X_remaining = X_ft[remaining_indices]
        # min distance from each remaining ID to any selected ID
        dists = np.linalg.norm(X_remaining[:, None, :] -
                               X_selected[None, :, :],
                               axis=2).min(axis=1)
        best_pos = int(np.argmax(dists))
        best_idx = remaining_indices[best_pos]
        selected_indices.append(best_idx)
        remaining_indices.remove(best_idx)

    return [ft_id_list[i] for i in selected_indices]


def greedy_coverage_ft_selection_2(
    X_joint_all: np.ndarray,
    id_array: np.ndarray,
    ft_ids: set,
    test_ids: set,
    n_select: int,
    blur: float,
    seed: int = 0,
    test_weight: float = 0.5,
) -> list:
    """
    Greedy coreset with Sinkhorn-based test proximity (computed once upfront)
    and L2-based coverage (cheap, per iteration).

    score(candidate) = (1 - test_weight) * L2_dist_to_selected   (coverage)
                     +      test_weight  * sinkhorn_proximity_to_test  (fixed)
    """
    ft_id_list = sorted(ft_ids)
    id_to_feat = {
        id_val: X_joint_all[id_array == id_val].mean(axis=0)
        for id_val in ft_id_list if (id_array == id_val).sum() >= 1
    }
    ft_id_list = list(id_to_feat.keys())
    X_ft = np.stack([id_to_feat[i] for i in ft_id_list])  # (n_ft, d)

    test_mask = np.isin(id_array, list(test_ids))
    X_test = X_joint_all[test_mask]

    # ── Sinkhorn distance from each FT ID to the test set — computed ONCE ────
    print(f"  Computing Sinkhorn proximity for {len(ft_id_list)} FT IDs...")
    sinkhorn_dist_to_test = np.zeros(len(ft_id_list))
    for i, id_val in enumerate(tqdm(ft_id_list, desc="Sinkhorn to test")):
        X_id = X_joint_all[id_array == id_val]
        if len(X_id) < 2:
            sinkhorn_dist_to_test[i] = np.inf
        else:
            sinkhorn_dist_to_test[i] = _sinkhorn_distance(X_id,
                                                          X_test,
                                                          blur=blur,
                                                          seed=seed)

    # Proximity = inverse distance (higher = closer to test)
    sinkhorn_proximity = 1.0 / (1.0 + sinkhorn_dist_to_test)  # (n_ft,)

    # Seed: FT ID with smallest Sinkhorn distance to test set
    first_idx = int(np.argmin(sinkhorn_dist_to_test))
    selected_indices = [first_idx]
    remaining_indices = [i for i in range(len(ft_id_list)) if i != first_idx]

    print(f"  Seed ID: {ft_id_list[first_idx]} "
          f"(Sinkhorn={sinkhorn_dist_to_test[first_idx]:.4f})")

    # ── Greedy iterations — L2 coverage + fixed Sinkhorn proximity ───────────
    for _ in range(n_select - 1):
        if not remaining_indices:
            break

        X_selected = X_ft[selected_indices]
        X_remaining = X_ft[remaining_indices]

        # L2 coverage: min dist from each remaining ID to any selected ID
        dists_to_selected = np.linalg.norm(X_remaining[:, None, :] -
                                           X_selected[None, :, :],
                                           axis=2).min(axis=1)

        # Normalise both terms to [0, 1]
        cov = dists_to_selected / (dists_to_selected.max() + 1e-8)
        prox = sinkhorn_proximity[remaining_indices]
        prox = prox / (prox.max() + 1e-8)

        score = (1 - test_weight) * cov + test_weight * prox
        best_pos = int(np.argmax(score))
        best_idx = remaining_indices[best_pos]

        selected_indices.append(best_idx)
        remaining_indices.remove(best_idx)

    return [ft_id_list[i] for i in selected_indices]


def greedy_coverage_ft_selection_no_seed(
    X_joint_all,
    id_array,
    ft_ids,
    n_select,
    seed=0,
):
    """Pure farthest-point coreset with random seed — no test set used."""
    ft_id_list = sorted(ft_ids)
    id_to_feat = {
        id_val: X_joint_all[id_array == id_val].mean(axis=0)
        for id_val in ft_id_list if (id_array == id_val).sum() >= 1
    }
    ft_id_list = list(id_to_feat.keys())
    X_ft = np.stack([id_to_feat[i] for i in ft_id_list])

    rng = np.random.default_rng(seed)
    first_idx = int(rng.integers(0, len(ft_id_list)))
    selected_indices = [first_idx]
    remaining_indices = [i for i in range(len(ft_id_list)) if i != first_idx]

    for _ in range(n_select - 1):
        if not remaining_indices:
            break
        X_selected = X_ft[selected_indices]
        X_remaining = X_ft[remaining_indices]
        dists = np.linalg.norm(X_remaining[:, None, :] -
                               X_selected[None, :, :],
                               axis=2).min(axis=1)
        best_idx = remaining_indices[int(np.argmax(dists))]
        selected_indices.append(best_idx)
        remaining_indices.remove(best_idx)

    return [ft_id_list[i] for i in selected_indices]

In [ ]:
# ── Build proximity rankings (Sinkhorn per ID vs test set) ────────────────────
FORCE_RECOMPUTE_RANK = False

ft_id_rankings = {}
for region in FT_RANK_REGIONS:
    save_path = BASE_DIR / f"ft_id_ranking_{region}.csv"

    df_monthly = res_xreg_by_source[region]["data_monthly"]
    winter_ids = df_monthly[df_monthly["PERIOD"] == "winter"]["ID"].unique().tolist()
    annual_ids = df_monthly[df_monthly["PERIOD"] == "annual"]["ID"].unique().tolist()
    rng = np.random.default_rng(cfg.seed)
    rng.shuffle(winter_ids)
    rng.shuffle(annual_ids)
    n_ft_w = max(1, int(len(winter_ids) * FT_FRAC)) if winter_ids else 0
    n_ft_a = max(1, int(len(annual_ids) * FT_FRAC))
    ft_ids  = winter_ids[:n_ft_w] + annual_ids[:n_ft_a]
    test_ids = winter_ids[n_ft_w:] + annual_ids[n_ft_a:]

    if not FORCE_RECOMPUTE_RANK and save_path.exists():
        ft_id_rankings[region] = pd.read_csv(save_path)
        print(f"Loaded ranking for {region} from {save_path}")
        continue

    Xm_all      = scaler_m.transform(df_monthly[MONTHLY_COLS].to_numpy(dtype=np.float64))
    Xs_id       = scaler_s.transform(_stake_topo(df_monthly))
    id_codes    = pd.Categorical(df_monthly["ID"]).codes
    X_joint_all = np.hstack([Xm_all, Xs_id[id_codes]]).astype(np.float32)
    id_array    = df_monthly["ID"].values
    X_test      = X_joint_all[np.isin(id_array, list(test_ids))]

    blur = _estimate_blur(X_joint_all, X_joint_all,
                          blur_quantile_multiplier=0.1, seed=cfg.seed)
    print(f"  {region}: blur={blur:.4f}, {len(ft_ids)} FT IDs")

    records = []
    for id_val in tqdm(sorted(ft_ids), desc=f"Proximity ranking {region}"):
        X_id = X_joint_all[id_array == id_val]
        if len(X_id) < 2: continue
        records.append({
            'ID': id_val,
            'sinkhorn_dist_to_test': _sinkhorn_distance(X_id, X_test, blur=blur, seed=cfg.seed)
        })

    df_ranked = pd.DataFrame(records).sort_values(
        'sinkhorn_dist_to_test').reset_index(drop=False)
    df_ranked.to_csv(save_path, index=False)
    ft_id_rankings[region] = df_ranked
    print(f"  Ranked {len(df_ranked)} IDs")

In [ ]:
# ── Main experiment loop: proximity + random ────────────────────────
TRAIN_FLAG = False
FORCE_RECOMPUTE_DATASETS = False
RETRAIN_REGIONS = []
#FT_STRATEGIES = ["full", "adapter"]
FT_STRATEGIES = ["full"]

source_codes_pretrain = build_source_codes_for_dataset(
    assets_full["ds_train"],
    pd.concat(
        [res_xreg_by_source[r]["data_monthly_aug"] for r in SOURCE_REGIONS],
        ignore_index=False),
    source_col="SOURCE_CODE",
)

SELECTIONS = ["proximity", "random"]

ft_frac_results = {
    region: {
        selection: {
            ft_strat: {
                f: []
                for f in FRACS_FT
            }
            for ft_strat in FT_STRATEGIES
        }
        for selection in SELECTIONS
    }
    for region in FT_RANK_REGIONS
}

for region in FT_RANK_REGIONS:
    print(f"\n{'='*60}  {region}")

    df_monthly = res_xreg_by_source[region]["data_monthly"]
    df_monthly_aug = res_xreg_by_source[region]["data_monthly_aug"]
    head_pad = res_xreg_by_source[region]["months_head_pad"]
    tail_pad = res_xreg_by_source[region]["months_tail_pad"]

    force_ds = FORCE_RECOMPUTE_DATASETS or (region in RETRAIN_REGIONS)
    force_train = TRAIN_FLAG or (region in RETRAIN_REGIONS)

    winter_ids = df_monthly[df_monthly["PERIOD"] ==
                            "winter"]["ID"].unique().tolist()
    annual_ids = df_monthly[df_monthly["PERIOD"] ==
                            "annual"]["ID"].unique().tolist()
    rng = np.random.default_rng(cfg.seed)
    rng.shuffle(winter_ids)
    rng.shuffle(annual_ids)
    n_ft_w = max(1, int(len(winter_ids) * FT_FRAC)) if winter_ids else 0
    n_ft_a = max(1, int(len(annual_ids) * FT_FRAC))
    ft_ids = winter_ids[:n_ft_w] + annual_ids[:n_ft_a]
    test_ids = winter_ids[n_ft_w:] + annual_ids[n_ft_a:]
    all_ft_ids = set(ft_ids)

    ds_test = build_or_load_lstm_dataset_only(
        cfg=cfg,
        key=f"FD_to_{region}_IDsplit_TEST",
        df_loss=df_monthly[df_monthly["ID"].isin(test_ids)],
        df_full=df_monthly_aug[df_monthly_aug["ID"].isin(test_ids)],
        months_head_pad=head_pad,
        months_tail_pad=tail_pad,
        MONTHLY_COLS=MONTHLY_COLS,
        STATIC_COLS=STATIC_COLS,
        cache_dir=str(CACHE_DIR),
        force_recompute=force_ds,
        kind="test",
        show_progress=False,
    )

    proximity_ids = ft_id_rankings[region]["ID"].tolist()

    named_strategies = {
        "proximity": proximity_ids,
    }

    rng_random = np.random.default_rng(cfg.seed + 42)

    for frac in FRACS_FT:
        n_select = max(1, int(len(ft_ids) * frac))

        # ── Proximity strategy ────────────────────────────────────────────────
        for selection_name, ordered_ids in named_strategies.items():
            selected_ids = ordered_ids[:n_select]

            ds_ft = build_or_load_lstm_dataset_only(
                cfg=cfg,
                key=
                f"FD_to_{region}_FTrank_frac{int(frac*100):02d}_{selection_name}",
                df_loss=df_monthly[df_monthly["ID"].isin(selected_ids)],
                df_full=df_monthly_aug[df_monthly_aug["ID"].isin(
                    selected_ids)],
                months_head_pad=head_pad,
                months_tail_pad=tail_pad,
                MONTHLY_COLS=MONTHLY_COLS,
                STATIC_COLS=STATIC_COLS,
                cache_dir=str(CACHE_DIR),
                force_recompute=force_ds,
                kind="ft",
                show_progress=False,
            )
            ft_train_idx, ft_val_idx = mbm.data_processing.MBSequenceDataset.split_indices(
                len(ds_ft), val_ratio=0.2, seed=cfg.seed)
            ft_assets_region = {
                "ds_ft": ds_ft,
                "ds_test": ds_test,
                "ft_train_idx": ft_train_idx,
                "ft_val_idx": ft_val_idx,
            }

            for ft_strat in FT_STRATEGIES:
                model_filename = str(
                    models_dir_ft /
                    f"transformer_{ft_strat}_{region}_{selection_name}_frac{int(frac*100):02d}_{MODEL_DATE}.pt"
                )

                model_trained = finetune_transformer_on_target(
                    cfg=cfg,
                    model_pooled=model_foundation,
                    ds_ft=ds_ft,
                    ds_pooled_scaler=ds_pooled_scaler,
                    device=device,
                    best_params=best_params_gs,
                    model_filename=model_filename,
                    strategy=ft_strat,
                    epochs=50,
                    lr_factor=0.1,
                    force_retrain=force_train,
                    verbose=False,
                )

                ds_test_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
                    ds_test)
                test_dl = mbm.data_processing.MBSequenceDataset.make_test_loader(
                    ds_test_copy,
                    ds_pooled_scaler,
                    batch_size=128,
                    seed=cfg.seed)
                m, _ = model_trained.evaluate_with_preds(
                    device, test_dl, ds_test_copy)
                ft_frac_results[region][selection_name][ft_strat][frac].append(
                    m["RMSE_annual"])

        # ── Random baseline ───────────────────────────────────────────────────
        for s in range(N_RANDOM_SEEDS):
            random_ids = rng_random.choice(list(all_ft_ids),
                                           size=n_select,
                                           replace=False).tolist()

            ds_ft_rnd = build_or_load_lstm_dataset_only(
                cfg=cfg,
                key=
                f"FD_to_{region}_FTrank_frac{int(frac*100):02d}_random_s{s}",
                df_loss=df_monthly[df_monthly["ID"].isin(random_ids)],
                df_full=df_monthly_aug[df_monthly_aug["ID"].isin(random_ids)],
                months_head_pad=head_pad,
                months_tail_pad=tail_pad,
                MONTHLY_COLS=MONTHLY_COLS,
                STATIC_COLS=STATIC_COLS,
                cache_dir=str(CACHE_DIR),
                force_recompute=force_ds,
                kind="ft",
                show_progress=False,
            )
            ft_train_idx, ft_val_idx = mbm.data_processing.MBSequenceDataset.split_indices(
                len(ds_ft_rnd), val_ratio=0.2, seed=cfg.seed)
            ft_assets_region_rnd = {
                "ds_ft": ds_ft_rnd,
                "ds_test": ds_test,
                "ft_train_idx": ft_train_idx,
                "ft_val_idx": ft_val_idx,
            }

            for ft_strat in FT_STRATEGIES:
                model_filename = str(
                    models_dir_ft /
                    f"transformer_{ft_strat}_{region}_random_frac{int(frac*100):02d}_s{s}_{MODEL_DATE}.pt"
                )

                model_trained = finetune_transformer_on_target(
                    cfg=cfg,
                    model_pooled=model_foundation,
                    ds_ft=ds_ft_rnd,
                    ds_pooled_scaler=ds_pooled_scaler,
                    device=device,
                    best_params=best_params_gs,
                    model_filename=model_filename,
                    strategy=ft_strat,
                    epochs=50,
                    lr_factor=0.1,
                    force_retrain=force_train,
                    verbose=False,
                )

                ds_test_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
                    ds_test)
                test_dl = mbm.data_processing.MBSequenceDataset.make_test_loader(
                    ds_test_copy,
                    ds_pooled_scaler,
                    batch_size=128,
                    seed=cfg.seed)
                m, _ = model_trained.evaluate_with_preds(
                    device, test_dl, ds_test_copy)
                ft_frac_results[region]["random"][ft_strat][frac].append(
                    m["RMSE_annual"])

        print(
            f"  frac={frac:.0%} | "
            f"proximity/{FT_STRATEGIES[0]}={ft_frac_results[region]['proximity'][FT_STRATEGIES[0]][frac][0]:.3f}"
            f" | random_{FT_STRATEGIES[0]}={np.mean(ft_frac_results[region]['random'][FT_STRATEGIES[0]][frac]):.3f}"
        )

In [ ]:
STRATEGY_STYLES = {
    "proximity": {
        "color": NATURE_PALETTE["blue"],
        "ls": "-",
        "label": "Proximity (Sinkhorn)"
    },
    "random": {
        "color": NATURE_PALETTE["sky_blue"],
        "ls": "--",
        "label": "Random"
    },
}

n_regions = len(FT_RANK_REGIONS)
n_cols = int(np.ceil(n_regions / 2))
n_rows = 2

fig, axes = plt.subplots(n_rows, n_cols,
                         figsize=(5 * n_cols, 4 * n_rows),
                         sharey=False)

# Always work with a flat list of axes
axes_flat = np.array(axes).flatten()

# Hide any unused subplots
for ax in axes_flat[n_regions:]:
    ax.set_visible(False)

finetuning = 'full'
for ax, region in zip(axes_flat, FT_RANK_REGIONS):
    frac_pct = [f * 100 for f in FRACS_FT]

    # Zero-shot baseline
    ds_test_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
        ft_assets_id[region]["ds_test"])
    test_dl = mbm.data_processing.MBSequenceDataset.make_test_loader(
        ds_test_copy, ds_pooled_scaler, batch_size=128, seed=cfg.seed)
    m_zs, _ = model_foundation.evaluate_with_preds(device, test_dl, ds_test_copy)

    # Random: mean ± std band
    rnd_means = [np.mean(ft_frac_results[region]["random"][finetuning][f]) for f in FRACS_FT]
    rnd_stds  = [np.std(ft_frac_results[region]["random"][finetuning][f])  for f in FRACS_FT]
    sty = STRATEGY_STYLES["random"]
    ax.fill_between(frac_pct,
                    np.array(rnd_means) - np.array(rnd_stds),
                    np.array(rnd_means) + np.array(rnd_stds),
                    alpha=0.15, color=sty["color"])
    ax.plot(frac_pct, rnd_means, f'o{sty["ls"]}',
            color=sty["color"], linewidth=0.8, markersize=3, label=sty["label"])

    # Proximity
    sty = STRATEGY_STYLES["proximity"]
    vals = [ft_frac_results[region]["proximity"][finetuning][f][0] for f in FRACS_FT]
    ax.plot(frac_pct, vals, f'o{sty["ls"]}',
            color=sty["color"], linewidth=0.8, markersize=3, label=sty["label"])

    ax.set_xlabel("FT data fraction (%)", fontsize=NATURE_SPECS["font_max_pt"])
    ax.set_ylabel("RMSE annual (m w.e.)", fontsize=NATURE_SPECS["font_max_pt"])
    #ax.set_title(region, fontsize=NATURE_SPECS["font_max_pt"] + 1, fontweight="bold")
    df_monthly = res_xreg_by_source[region]["data_monthly"]
    winter_ids = df_monthly[df_monthly["PERIOD"] == "winter"]["ID"].unique()
    annual_ids = df_monthly[df_monthly["PERIOD"] == "annual"]["ID"].unique()
    n_ft_w = max(1, int(len(winter_ids) * FT_FRAC)) if len(winter_ids) > 0 else 0
    n_ft_a = max(1, int(len(annual_ids) * FT_FRAC))
    n_test_ids = (len(winter_ids) - n_ft_w) + (len(annual_ids) - n_ft_a)
    ax.set_title(f"{region}\n(n={n_test_ids} holdout IDs)",
                 fontsize=NATURE_SPECS["font_max_pt"] + 1, fontweight="bold")
    ax.legend(fontsize=NATURE_SPECS["font_max_pt"] - 1)
    apply_nature_style(ax)

fig.suptitle("Fine-tuning data selection: proximity vs random (ID-level)",
             fontsize=NATURE_SPECS["font_max_pt"] + 2, fontweight="bold")
fig.tight_layout()
fig.savefig(f"figures/ft_selection_proximity_vs_random_{FT_STRATEGY}.png",
            dpi=NATURE_SPECS["dpi"], bbox_inches="tight")
plt.show()